In [1]:
# ==========================================
# 实验：二分类（只保留前两类）+ fold内归一化
# 目的：验证数据子集是否为关键因素
# ==========================================

import numpy as np
import pandas as pd
import pickle
import os
import time
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.linear_model import LogisticRegressionCV
from sklearn.preprocessing import MinMaxScaler
from sktime.transformations.panel.rocket import MiniRocketMultivariate
import warnings
warnings.filterwarnings('ignore')

print("="*60)
print("NGAFID 二分类实验（前两类 + fold内归一化）")
print("="*60)

# ==========================================
# 1. 加载数据
# ==========================================
data_dir = '/root'

with open(os.path.join(data_dir, 'flight_data.pkl'), 'rb') as f:
    data = pickle.load(f)

header_df = pd.read_csv(os.path.join(data_dir, 'flight_header.csv'))

print(f"\n✅ 数据加载成功")
print(f"  航班数: {len(data)}")
print(f"  header记录数: {len(header_df)}")

# ==========================================
# 2. 筛选：只保留前两类
# ==========================================
main_classes = ['intake gasket leak/damage', 'rocker cover leak/loose/damage']

mask = (abs(header_df['date_diff']) <= 2) & (header_df['date_diff'] != 0)
mask = mask & (header_df['label'].isin(main_classes))

filtered_header = header_df[mask].copy()
filtered_header = filtered_header.reset_index(drop=True)
flight_ids = filtered_header['Master Index'].values

print(f"\n📊 筛选后（只保留前两类）:")
print(f"  航班数: {len(filtered_header)}")
print(f"  维护后 (0): {sum(filtered_header['before_after']==0)}")
print(f"  维护前 (1): {sum(filtered_header['before_after']==1)}")
print(f"  类别分布:")
for cls in main_classes:
    count = sum(filtered_header['label'] == cls)
    print(f"    {cls}: {count}")

# ==========================================
# 3. 准备特征和标签（不预先归一化）
# ==========================================
target_len = 4096
X_list = []
y = []

print("\n⏳ 准备数据（不预先归一化）...")

for idx, flight_id in enumerate(flight_ids):
    sensor_data = data[flight_id]
    sensor_data = np.nan_to_num(sensor_data, nan=0.0)
    
    # 只截取，不归一化
    if sensor_data.shape[0] >= target_len:
        sensor_data = sensor_data[-target_len:, :]
    else:
        pad_width = ((0, target_len - sensor_data.shape[0]), (0, 0))
        sensor_data = np.pad(sensor_data, pad_width, mode='constant', constant_values=0)
    
    X_list.append(sensor_data)
    label = filtered_header.iloc[idx]['before_after']
    y.append(label)

X = np.array(X_list, dtype=np.float32)
y = np.array(y)

print(f"\n✅ 数据准备完成")
print(f"  X 形状: {X.shape}")
print(f"  标签分布: 0={sum(y==0)}, 1={sum(y==1)}")

# ==========================================
# 4. 5折交叉验证（fold内归一化）
# ==========================================
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

minirocket = MiniRocketMultivariate(
    random_state=42,
    num_kernels=10000,
)

classifier = LogisticRegressionCV(
    Cs=10, 
    cv=3, 
    random_state=42, 
    max_iter=5000
)

accuracies, f1_scores, auc_scores = [], [], []

print("\n" + "="*60)
print("5折交叉验证（前两类 + fold内归一化）")
print("="*60)

fold = 1
for train_idx, val_idx in skf.split(X, y):
    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]
    
    print(f"\n--- Fold {fold} ---")
    print(f"  训练集: {len(train_idx)} 样本 (0={sum(y_train==0)}, 1={sum(y_train==1)})")
    print(f"  验证集: {len(val_idx)} 样本 (0={sum(y_val==0)}, 1={sum(y_val==1)})")
    
    start = time.time()
    
    # fold内归一化
    n_samples, length, channels = X_train.shape
    X_train_flat = X_train.reshape(-1, channels)
    scaler = MinMaxScaler()
    scaler.fit(X_train_flat)
    
    X_train_norm = scaler.transform(X_train_flat).reshape(n_samples, length, channels)
    X_val_flat = X_val.reshape(-1, channels)
    X_val_norm = scaler.transform(X_val_flat).reshape(X_val.shape)
    
    # MiniRocket 特征提取
    X_train_transform = minirocket.fit_transform(X_train_norm, y_train)
    X_val_transform = minirocket.transform(X_val_norm)
    
    # 训练分类器
    classifier.fit(X_train_transform, y_train)
    
    # 预测
    y_pred = classifier.predict(X_val_transform)
    y_prob = classifier.predict_proba(X_val_transform)[:, 1]
    
    elapsed = time.time() - start
    
    acc = accuracy_score(y_val, y_pred)
    f1 = f1_score(y_val, y_pred)
    auc = roc_auc_score(y_val, y_prob)
    
    accuracies.append(acc)
    f1_scores.append(f1)
    auc_scores.append(auc)
    
    print(f"  准确率: {acc:.4f}, F1: {f1:.4f}, AUC: {auc:.4f}, 耗时: {elapsed:.2f}s")
    fold += 1

print("\n" + "="*60)
print("📊 二分类实验最终结果:")
print("="*60)
print(f"  准确率: {np.mean(accuracies):.4f} ± {np.std(accuracies):.4f}")
print(f"  F1分数: {np.mean(f1_scores):.4f} ± {np.std(f1_scores):.4f}")
print(f"  AUC:    {np.mean(auc_scores):.4f} ± {np.std(auc_scores):.4f}")
print("="*60)

# ==========================================
# 6. 结果对比
# ==========================================
print("\n📊 实验对比:")
print("-" * 60)
print(f"  完整19类 + 全局归一化:     ~54.3%")
print(f"  完整19类 + fold内归一化:   58.7%")
print(f"  前两类 + fold内归一化:     {np.mean(accuracies):.4f}")
print("-" * 60)

if np.mean(accuracies) > 0.65:
    print("\n🎉 显著提升！数据子集是关键因素！")
else:
    print("\n⚠️ 提升有限，需要尝试其他方向")

NGAFID 二分类实验（前两类 + fold内归一化）

✅ 数据加载成功
  航班数: 11446
  header记录数: 11446

📊 筛选后（只保留前两类）:
  航班数: 6570
  维护后 (0): 3448
  维护前 (1): 3122
  类别分布:
    intake gasket leak/damage: 4310
    rocker cover leak/loose/damage: 2260

⏳ 准备数据（不预先归一化）...

✅ 数据准备完成
  X 形状: (6570, 4096, 23)
  标签分布: 0=3448, 1=3122

5折交叉验证（前两类 + fold内归一化）

--- Fold 1 ---
  训练集: 5256 样本 (0=2759, 1=2497)
  验证集: 1314 样本 (0=689, 1=625)
  准确率: 0.5997, F1: 0.5602, AUC: 0.6394, 耗时: 68.72s

--- Fold 2 ---
  训练集: 5256 样本 (0=2759, 1=2497)
  验证集: 1314 样本 (0=689, 1=625)
  准确率: 0.5997, F1: 0.5572, AUC: 0.6454, 耗时: 68.40s

--- Fold 3 ---
  训练集: 5256 样本 (0=2758, 1=2498)
  验证集: 1314 样本 (0=690, 1=624)
  准确率: 0.6043, F1: 0.5616, AUC: 0.6405, 耗时: 67.31s

--- Fold 4 ---
  训练集: 5256 样本 (0=2758, 1=2498)
  验证集: 1314 样本 (0=690, 1=624)
  准确率: 0.6256, F1: 0.5852, AUC: 0.6650, 耗时: 67.92s

--- Fold 5 ---
  训练集: 5256 样本 (0=2758, 1=2498)
  验证集: 1314 样本 (0=690, 1=624)
  准确率: 0.5959, F1: 0.5542, AUC: 0.6268, 耗时: 66.57s

📊 二分类实验最终结果:
  准确率: 0.6050 ± 0.0106
 